# 🧱 Modular Photonics Compute — Build Anything

> **Theme:** every section is a self-contained module with clean inputs/outputs.
> Snap them together like LEGO. The meta-pattern applies to hardware, software, optics, math.

| § | Module | Boss concept |
|---|---|---|
| §1 | Project Modularity | Abstract factory · DAG · plugin registry |
| §2 | Verilog + Waveform Physics | RTL → netlist → waveform · signal integrity |
| §3 | Photonic AI Datacenter | WDM · bandwidth density · energy/bit |
| §4 | Vector Core / GEMM | Systolic array · roofline · tiling |
| §5 | 300k–500k Trajectory Integrals | C struct → dataclass · vectorized RK4 |
| §6 | Free-Space EM Structs | Gaussian beam ABCD · diffraction integral |
| §7 | Lasers + Spectroscopy | Voigt lineshape · Beer-Lambert · TOP500 HPC |
| §8 | Fixed Points + Equilibrium | Phase portrait · Lyapunov · tilt stability |
| §9 | IP/DNS/DOM Agents | Socket DNS · network topo · DOM parse |

In [ ]:
import numpy as np
import sympy as sp
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrow, Rectangle, FancyArrowPatch
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Any, Optional, Tuple
from scipy.integrate import solve_ivp, quad
from scipy.signal import butter, filtfilt
from scipy.special import voigt_profile, wofz
import socket, struct as _struct, itertools, time, abc, hashlib, warnings, re
warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size':10,'figure.dpi':100})
np.random.seed(2026)
print('Modular Photonics Compute — all modules loaded OK')

---
## §1 🧱 Project Modularity — Build Anything

**Pattern:** every physics/compute module implements one interface:

```python
class Module(ABC):
    @abstractmethod
    def run(self, inputs: dict) -> dict: ...
    @abstractmethod
    def describe(self) -> str: ...
```

**DAG execution:** topological sort → run in dependency order → cache outputs.

**Plugin registry:** `@register('name')` decorator → `registry['name']()` → instantiate.

**Why it matters for this repo:** `gs_core.py`, `gs_fno.py`, each notebook section,
each firmware driver — all become swappable modules with the same interface.

In [ ]:
# §1 — Modular architecture implementation

import abc
from collections import defaultdict, deque

# ── Abstract base ──────────────────────────────────────────────────
class Module(abc.ABC):
    @abc.abstractmethod
    def run(self, inputs: dict) -> dict: ...
    @abc.abstractmethod
    def describe(self) -> str: ...

# ── Plugin registry ────────────────────────────────────────────────
_REGISTRY: Dict[str,type] = {}

def register(name: str):
    def decorator(cls):
        _REGISTRY[name] = cls
        return cls
    return decorator

# ── DAG Pipeline ───────────────────────────────────────────────────
class Pipeline:
    def __init__(self):
        self.nodes: Dict[str, Module]  = {}
        self.edges: Dict[str, List[str]] = defaultdict(list)  # name→deps

    def add(self, name: str, module: Module, deps: List[str] = None):
        self.nodes[name] = module
        if deps:
            self.edges[name].extend(deps)

    def topo_sort(self) -> List[str]:
        in_deg = {n:0 for n in self.nodes}
        for n,deps in self.edges.items():
            for d in deps:
                in_deg[n] += 1
        q  = deque(n for n,d in in_deg.items() if d==0)
        out= []
        while q:                            # loop: topo sort
            n = q.popleft(); out.append(n)
            for m,deps in self.edges.items():
                if n in deps:
                    in_deg[m] -= 1
                    if in_deg[m]==0: q.append(m)
        return out

    def run(self, inputs: dict={}) -> dict:
        cache = dict(inputs)
        for name in self.topo_sort():       # loop: execute in order
            result = self.nodes[name].run(cache)
            cache.update(result)
            cache[f'__{name}__done'] = True
        return cache

# ── Concrete modules ───────────────────────────────────────────────
@register('gaussian_beam')
class GaussianBeamModule(Module):
    def run(self, inp):
        w0    = inp.get('w0', 100e-6)
        lam   = inp.get('wavelength', 1550e-9)
        z_arr = inp.get('z_arr', np.linspace(-0.1,0.1,200))
        zR    = np.pi*w0**2/lam
        w_z   = w0*np.sqrt(1+(z_arr/zR)**2)
        return {'beam_radius': w_z, 'rayleigh_range': zR, 'z_arr': z_arr}
    def describe(self): return 'Gaussian beam propagation: w(z) = w0*sqrt(1+(z/zR)^2)'

@register('beer_lambert')
class BeerLambertModule(Module):
    def run(self, inp):
        alpha = inp.get('absorption_coeff', 1000.0)  # /m
        L     = inp.get('path_length', 0.01)         # m
        I0    = inp.get('I0', 1.0)
        T     = np.exp(-alpha*L)
        return {'transmittance': T, 'absorbance': -np.log10(T)}
    def describe(self): return 'Beer-Lambert: T = exp(-alpha*L)'

@register('trajectory_integrator')
class TrajectoryModule(Module):
    def run(self, inp):
        n       = inp.get('n_particles', 1000)
        dt      = inp.get('dt', 1e-3)
        n_steps = inp.get('n_steps', 100)
        x0 = np.random.randn(n,3)*0.1
        v0 = np.random.randn(n,3)*0.5
        # RK4 step: harmonic oscillator F=-k*x
        k  = inp.get('k_spring', 1.0)
        def F(x): return -k*x
        x,v = x0.copy(), v0.copy()
        for _ in range(n_steps):           # loop: RK4
            k1v=F(x); k1x=v
            k2v=F(x+0.5*dt*k1x); k2x=v+0.5*dt*k1v
            k3v=F(x+0.5*dt*k2x); k3x=v+0.5*dt*k2v
            k4v=F(x+dt*k3x);     k4x=v+dt*k3v
            x  = x + dt/6*(k1x+2*k2x+2*k3x+k4x)
            v  = v + dt/6*(k1v+2*k2v+2*k3v+k4v)
        return {'final_positions': x, 'final_velocities': v}
    def describe(self): return 'RK4 trajectory integrator: F=-kx harmonic'

# ── Wire up a demo pipeline ────────────────────────────────────────
pipe = Pipeline()
pipe.add('beam',  _REGISTRY['gaussian_beam']())
pipe.add('traj',  _REGISTRY['trajectory_integrator'](), deps=['beam'])
pipe.add('absorb',_REGISTRY['beer_lambert']())

result = pipe.run({'w0':80e-6,'wavelength':1064e-9,'n_particles':500,
                   'absorption_coeff':5000,'path_length':0.005})

print('§1 Pipeline execution order:', pipe.topo_sort())
print(f'   Gaussian beam: zR = {result["rayleigh_range"]*1e3:.2f} mm')
print(f'   Beer-Lambert:  T  = {result["transmittance"]:.4f}  A = {result["absorbance"]:.3f}')
print(f'   Trajectory:    <|x|> final = {np.linalg.norm(result["final_positions"],axis=1).mean():.4f}')
print(f'\n   Registry has {len(_REGISTRY)} modules: {list(_REGISTRY.keys())}')

# Visualize DAG
fig, axes = plt.subplots(1,2,figsize=(12,4))
# Draw pipeline DAG
ax_dag = axes[0]
node_pos = {'beam':(0.5,0.8), 'absorb':(0.1,0.3), 'traj':(0.9,0.3)}
node_col = {'beam':'#4CAF50','absorb':'#2196F3','traj':'#FF9800'}
for n,(x,y) in node_pos.items():
    ax_dag.add_patch(Rectangle((x-0.12,y-0.07),0.24,0.14,
                                facecolor=node_col[n],alpha=0.8,edgecolor='k',lw=1.5))
    ax_dag.text(x,y,n,ha='center',va='center',fontsize=9,fontweight='bold',color='white')
# edges
for n,deps in pipe.edges.items():
    nx,ny = node_pos[n]
    for d in deps:
        dx,dy = node_pos[d]
        ax_dag.annotate('',xy=(nx,ny+0.07),xytext=(dx,dy-0.07),
                        arrowprops=dict(arrowstyle='->', color='k', lw=1.5))
ax_dag.set_xlim(0,1); ax_dag.set_ylim(0,1); ax_dag.axis('off')
ax_dag.set_title('§1 Module DAG (topological order)')
ax_dag.text(0.5,0.02,'Execution: beam → absorb, traj',ha='center',fontsize=8,style='italic')

# Module interface diagram
ax_if = axes[1]
ax_if.axis('off')
module_desc = [
    ('Input dict', '#E3F2FD', 0.85),
    ('Module.run()', '#1565C0', 0.65),
    ('Output dict', '#E8F5E9', 0.45),
    ('Cache merge', '#F3E5F5', 0.25),
]
for label,col,y in module_desc:
    ax_if.add_patch(Rectangle((0.2,y-0.06),0.6,0.12,facecolor=col,
                               edgecolor='k',lw=1.5,alpha=0.9))
    ax_if.text(0.5,y,label,ha='center',va='center',fontsize=10,
               color='white' if 'Module' in label else 'black', fontweight='bold')
    if y>0.26:
        ax_if.annotate('',xy=(0.5,y-0.08),xytext=(0.5,y-0.14),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax_if.set_title('§1 Module interface contract')

# Plugin registry table
reg_text = 'Plugin Registry:\n' + '\n'.join(f'  [{k}] → {v.__name__}' for k,v in _REGISTRY.items())
ax_if.text(0.05,0.12,reg_text,transform=ax_if.transAxes,fontsize=8,
           fontfamily='monospace',va='bottom',
           bbox=dict(boxstyle='round',facecolor='#FFF9C4',alpha=0.9))
plt.suptitle('§1 Modular Architecture — Build Anything', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §2 💻 Verilog + Compiler Wave Physics — RTL → Netlist → Waveform

**§2.1 Digital signal integrity:**
$V_{noise} = L_{pkg}\frac{dI}{dt}$, crosstalk $V_{XT} = C_m \frac{dV}{dt} Z_0$

**§2.2 Transmission line (lossy):**
$\gamma = \sqrt{(R+j\omega L)(G+j\omega C)}$, $Z_0 = \sqrt{(R+j\omega L)/(G+j\omega C)}$

**§2.3 Eye diagram** — CDR threshold, jitter, ISI

**§2.4 Verilog synthesis pipeline:**
RTL `.v` → `yosys` → netlist JSON → `nextpnr` → bitstream → FPGA

In [ ]:
# §2 — Verilog + waveform physics

# §2.1 Simultaneous switching noise (SSN)
n_bits = 32; L_pkg = 2e-9; I_per_bit = 20e-3; t_rise = 200e-12
dI_dt  = n_bits * I_per_bit / t_rise
V_ssn  = L_pkg * dI_dt
print(f'§2.1 SSN: L={L_pkg*1e9:.0f}nH, dI/dt={dI_dt:.1f}A/s → V_noise={V_ssn:.3f}V')

# §2.2 Transmission line: 50Ω, FR4 at 10 GHz
f_arr  = np.logspace(6, 11, 500)   # 1 MHz to 100 GHz
omega  = 2*np.pi*f_arr
R_tl   = 0.5    # Ω/m (skin effect simplified)
L_tl   = 400e-12 # H/m
G_tl   = 0.0    # S/m (lossless dielectric approx)
C_tl   = 160e-12 # F/m (εr=4.2 FR4)
gamma_c = np.sqrt((R_tl + 1j*omega*L_tl)*(G_tl + 1j*omega*C_tl))
alpha_c = np.real(gamma_c)   # attenuation (Np/m)
beta_c  = np.imag(gamma_c)   # phase (rad/m)
Z0_c    = np.sqrt((R_tl+1j*omega*L_tl)/(G_tl+1j*omega*C_tl+1e-30))
Z0_mag  = np.abs(Z0_c)
# Insertion loss in dB for 5 cm trace
IL_dB   = -20*np.log10(np.exp(-alpha_c*0.05))
print(f'§2.2 TL at 10 GHz: α={alpha_c[400]:.2f}Np/m, Z0={Z0_mag[400]:.1f}Ω, IL(5cm)={IL_dB[400]:.2f}dB')

# §2.3 Eye diagram: 10 Gbps NRZ with ISI + jitter
bit_rate = 10e9; T_bit = 1/bit_rate; n_bits_eye = 2000
# Random NRZ bits
bits = np.random.choice([-1.0,1.0], size=n_bits_eye)
sps  = 64   # samples per symbol
t_full = np.arange(n_bits_eye*sps)/bit_rate/sps

# Raised-cosine pulse shaping (beta=0.3)
beta_rc = 0.35; t_rc = np.arange(-4,4,1/sps)*T_bit
rc_filter= np.sinc(t_rc/T_bit)*np.cos(np.pi*beta_rc*t_rc/T_bit) / \
           np.maximum(1-(2*beta_rc*t_rc/T_bit)**2, 1e-6)
rc_filter/=rc_filter.sum()

signal_src = np.zeros(n_bits_eye*sps)
for i,b in enumerate(bits):         # loop: upsample
    signal_src[i*sps] = b
signal_shaped = np.convolve(signal_src, rc_filter, mode='same')

# Attenuation at Nyquist (channel lowpass)
b_lp,a_lp = butter(2, 0.35, btype='low')
signal_rx  = filtfilt(b_lp, a_lp, signal_shaped)

# Add Gaussian jitter and noise
jitter_ps    = 8e-12  # 8 ps rms jitter
jitter_samps = int(jitter_ps / (T_bit/sps))
noise        = np.random.randn(len(signal_rx))*0.05
signal_rx   += noise

# Build eye diagram: fold at 2*T_bit
eye_period = 2*sps; n_traces = 400
eye_traces  = []
start_idx   = sps*10
for i in range(n_traces):            # loop: eye traces
    idx = start_idx + i*sps + np.random.randint(-jitter_samps, jitter_samps+1)
    if idx+eye_period < len(signal_rx):
        eye_traces.append(signal_rx[idx:idx+eye_period])
eye_arr = np.array(eye_traces)
t_eye   = np.linspace(0, 2*T_bit, eye_period)*1e12  # ps

# §2.4 Verilog synthesis text pipeline
synth_flow = [
    ('RTL (Verilog)', 'adder4.v, alu.v', '#E3F2FD'),
    ('Elaboration',   'yosys read_verilog', '#BBDEFB'),
    ('Logic Synth',   'yosys synth -top', '#90CAF9'),
    ('Tech Map',       'abc -liberty sky130', '#64B5F6'),
    ('Netlist JSON',  'write_json netlist.json', '#42A5F5'),
    ('Place & Route', 'nextpnr-ice40', '#2196F3'),
    ('Bitstream',     'icepack → .bin', '#1565C0'),
    ('FPGA Program',  'iceprog / openFPGALoader', '#0D47A1'),
]

# ── Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15,8))
gs  = gridspec.GridSpec(2,4,fig)

# TL attenuation vs frequency
ax_tl = fig.add_subplot(gs[0,0])
ax_tl.semilogx(f_arr/1e9, IL_dB,'b-',lw=2)
ax_tl.axvline(10,ls='--',color='r',lw=1.5,label='10 GHz')
ax_tl.set_xlabel('Frequency (GHz)'); ax_tl.set_ylabel('Insertion Loss (dB)')
ax_tl.set_title('§2.2 FR4 trace IL (5cm)')
ax_tl.legend(fontsize=8); ax_tl.grid(True,alpha=0.3)

# Z0 vs frequency
ax_z0 = fig.add_subplot(gs[0,1])
ax_z0.semilogx(f_arr/1e9, Z0_mag,'g-',lw=2)
ax_z0.axhline(50,ls='--',color='k',lw=1,label='50Ω target')
ax_z0.set_xlabel('Frequency (GHz)'); ax_z0.set_ylabel('|Z0| (Ω)')
ax_z0.set_title('§2.2 Transmission line impedance')
ax_z0.legend(fontsize=8); ax_z0.grid(True,alpha=0.3)

# Eye diagram
ax_eye = fig.add_subplot(gs[0,2:])
for tr in eye_arr[:200]:             # loop: draw eye
    ax_eye.plot(t_eye, tr, 'b-', alpha=0.03, lw=0.8)
ax_eye.axhline(0,color='r',ls='--',lw=1.5,label='Decision threshold')
ax_eye.axvline(T_bit*1e12,color='gray',ls=':',lw=1)
ax_eye.set_xlabel('Time (ps)'); ax_eye.set_ylabel('Amplitude')
ax_eye.set_title(f'§2.3 Eye diagram — 10Gbps NRZ, jitter={jitter_ps*1e12:.0f}ps rms')
ax_eye.legend(fontsize=8)

# Synthesis flow
ax_sf = fig.add_subplot(gs[1,:2])
ax_sf.axis('off')
for i,(label,detail,color) in enumerate(synth_flow):  # loop: synthesis stages
    y = 0.9 - i*0.11
    ax_sf.add_patch(Rectangle((0.02,y-0.045),0.96,0.09,
                                facecolor=color,edgecolor='k',lw=1))
    ax_sf.text(0.30,y,label,ha='center',va='center',fontsize=8,fontweight='bold',
               color='white' if i>=4 else 'black')
    ax_sf.text(0.70,y,detail,ha='center',va='center',fontsize=7,
               color='white' if i>=4 else '#333')
    if i < len(synth_flow)-1:
        ax_sf.annotate('',xy=(0.5,y-0.05),xytext=(0.5,y-0.045),
                        arrowprops=dict(arrowstyle='->', color='k',lw=1.5))
ax_sf.set_title('§2.4 Verilog → FPGA Synthesis Flow')

# SSN vs switching bits
ax_ssn = fig.add_subplot(gs[1,2:])
n_bits_arr = np.arange(1,65)
V_ssn_arr  = n_bits_arr * I_per_bit / t_rise * np.array([0.5e-9,1e-9,2e-9,5e-9])[:,None]
for L_val,V_arr in zip([0.5,1,2,5]*[1e-9],V_ssn_arr):
    ax_ssn.plot(n_bits_arr, V_arr*1000, lw=2, label=f'L={L_val*1e9:.1f}nH')
ax_ssn.axhline(100,ls='--',color='r',lw=1.5,label='100mV noise budget')
ax_ssn.set_xlabel('Switching bits N'); ax_ssn.set_ylabel('V_SSN (mV)')
ax_ssn.set_title('§2.1 SSN vs switching count (FR4 PCB)')
ax_ssn.legend(fontsize=7); ax_ssn.grid(True,alpha=0.3)
plt.suptitle('💻 §2: Verilog + Waveform Physics', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 🔬 Photonic AI Datacenter — WDM · Bandwidth Density · Energy/bit

**§3.1 WDM capacity:** $C = N_\lambda \times B_{per\lambda}$, $N_\lambda$ channels on 100 GHz ITU grid

**§3.2 Bandwidth density:** $BD = C / A_{fiber}$ (Tb/s per mm²)

**§3.3 Photonic energy/bit:**
$E_{bit} = P_{laser} / R_{bit}$; target < 1 pJ/bit for datacenter interconnects

**§3.4 Silicon photonic link budget:**
$P_{rx} = P_{tx} - IL_{coupling} - IL_{wvg} - IL_{mux} - P_{split}$ (dB)

In [ ]:
# §3 — Photonic AI Datacenter

# §3.1 WDM system (C+L band)
c_light = 3e8
lam_c_start = 1530e-9; lam_c_end = 1565e-9   # C-band
lam_l_start = 1565e-9; lam_l_end = 1625e-9   # L-band
ch_spacing_GHz = 100  # ITU standard

def count_channels(lam_start, lam_end, spacing_GHz):
    f_start = c_light/lam_end
    f_end   = c_light/lam_start
    return int((f_end-f_start)/(spacing_GHz*1e9))

n_C = count_channels(lam_c_start, lam_c_end, ch_spacing_GHz)
n_L = count_channels(lam_l_start, lam_l_end, ch_spacing_GHz)
print(f'§3.1 WDM: C-band {n_C} ch, L-band {n_L} ch, total {n_C+n_L} channels @ 100GHz ITU')

# Modulation formats: capacity per channel
mod_formats = {
    'NRZ':        {'bits_per_sym':1, 'baud':28e9,  'margin_dB':5},
    'PAM4':       {'bits_per_sym':2, 'baud':28e9,  'margin_dB':3},
    'DP-QPSK':    {'bits_per_sym':4, 'baud':32e9,  'margin_dB':4},
    'DP-16QAM':   {'bits_per_sym':8, 'baud':32e9,  'margin_dB':2},
    'DP-64QAM':   {'bits_per_sym':12,'baud':96e9,  'margin_dB':1},
}
for mf,p in mod_formats.items():
    cap_ch   = p['bits_per_sym']*p['baud']/1e9
    cap_tot  = cap_ch*(n_C+n_L)/1e3
    print(f'  {mf:12s}: {cap_ch:.0f} Gb/s/ch → {cap_tot:.1f} Tb/s total')

# §3.3 Energy per bit for photonic link
P_laser_dBm = 10.0    # 10 mW
P_laser_W   = 10**(P_laser_dBm/10)*1e-3
IL_coupling  = 1.5    # dB (fiber-chip)
IL_waveguide = 1.0    # dB (Si wvg 2cm)
IL_mux       = 2.5    # dB (AWG mux)
IL_mod       = 5.0    # dB (modulator insertion)
sens_dBm     = -18.0  # receiver sensitivity
P_rx_dBm     = P_laser_dBm - IL_coupling - IL_waveguide - IL_mux - IL_mod
margin_dB    = P_rx_dBm - sens_dBm
print(f'\n§3.4 Link budget: P_tx={P_laser_dBm}dBm → P_rx={P_rx_dBm:.1f}dBm, margin={margin_dB:.1f}dB')

bit_rates = np.logspace(9, 13, 200)   # 1 Gb/s to 10 Tb/s
P_laser_range = np.array([1e-3, 5e-3, 10e-3, 50e-3])  # 1-50 mW
E_bit = {}
for P in P_laser_range:               # loop: energy/bit curves
    WPE   = 0.25  # wall-plug efficiency
    P_elec= P/WPE
    E_bit[P*1e3] = P_elec/bit_rates   # J/bit

# §3.2 Bandwidth density comparison
interconnect_tech = {
    'Cu PCIe4':        {'BW_Tbps':0.256, 'Area_mm2':50,  'color':'gray'},
    'Cu SerDes 112G':  {'BW_Tbps':0.448, 'Area_mm2':30,  'color':'brown'},
    'Si Ph 1550nm':    {'BW_Tbps':1.6,   'Area_mm2':2,   'color':'blue'},
    'Si Ph C+L WDM':   {'BW_Tbps':12.8,  'Area_mm2':5,   'color':'green'},
    'InP PICs':        {'BW_Tbps':25.6,  'Area_mm2':10,  'color':'purple'},
    'Free-space opt':  {'BW_Tbps':100,   'Area_mm2':100, 'color':'orange'},
}
bd = {k: v['BW_Tbps']/(v['Area_mm2']/1e6) for k,v in interconnect_tech.items()}  # Tb/s/m²

# AI Datacenter: GPU cluster optical interconnect
n_gpus = np.array([8,64,512,4096,32768])
bw_per_gpu = 3.2e12   # 3.2 Tb/s NVLink4
total_bw_needed = n_gpus * bw_per_gpu / 2  # bisection bandwidth
n_fibers_needed = total_bw_needed / (6.4e12)  # 6.4 Tb/s per fiber (C+L WDM)

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# Energy per bit
for P_mw,E_arr in E_bit.items():     # loop: energy curves
    axes[0][0].loglog(bit_rates/1e9, E_arr*1e12, lw=2, label=f'P={P_mw:.0f}mW')
axes[0][0].axhline(1,ls='--',color='r',lw=2,label='1 pJ/bit target')
axes[0][0].set_xlabel('Bit rate (Gb/s)'); axes[0][0].set_ylabel('Energy/bit (pJ)')
axes[0][0].set_title('§3.3 Photonic link energy/bit (WPE=25%)')
axes[0][0].legend(fontsize=7); axes[0][0].grid(True,alpha=0.3)

# Bandwidth density
names_bd = list(bd.keys()); vals_bd = list(bd.values())
colors_bd= [v['color'] for v in interconnect_tech.values()]
axes[0][1].barh(range(len(names_bd)), [v/1e6 for v in vals_bd],
                color=colors_bd, alpha=0.8, edgecolor='k')
axes[0][1].set_yticks(range(len(names_bd))); axes[0][1].set_yticklabels(names_bd,fontsize=8)
axes[0][1].set_xlabel('BW density (Tb/s/mm²)')
axes[0][1].set_title('§3.2 Bandwidth density comparison')
axes[0][1].set_xscale('log'); axes[0][1].grid(True,alpha=0.3,axis='x')

# Link budget waterfall
stages  = ['Laser','Coupling','Waveguide','AWG Mux','Modulator','Rx']
stage_P = [P_laser_dBm,
           P_laser_dBm-IL_coupling,
           P_laser_dBm-IL_coupling-IL_waveguide,
           P_laser_dBm-IL_coupling-IL_waveguide-IL_mux,
           P_rx_dBm, sens_dBm]
losses  = [0, IL_coupling, IL_waveguide, IL_mux, IL_mod, 0]
colors_lb= ['green','orange','orange','orange','orange','blue']
axes[0][2].bar(range(len(stages)), stage_P, color=colors_lb, alpha=0.8, edgecolor='k')
axes[0][2].axhline(sens_dBm,ls='--',color='r',lw=2,label=f'Rx sens={sens_dBm}dBm')
axes[0][2].set_xticks(range(len(stages))); axes[0][2].set_xticklabels(stages,fontsize=8)
axes[0][2].set_ylabel('Power (dBm)'); axes[0][2].set_title('§3.4 Si photonic link budget')
axes[0][2].legend(fontsize=8); axes[0][2].grid(True,alpha=0.3,axis='y')

# WDM channel map (C+L band)
f_C = np.arange(int(c_light/lam_c_end/1e9), int(c_light/lam_c_start/1e9), 100)  # GHz
f_L = np.arange(int(c_light/lam_l_end/1e9), int(c_light/lam_l_start/1e9), 100)
axes[1][0].vlines(f_C,0,1,'b',lw=1.5,label=f'C-band ({len(f_C)}ch)')
axes[1][0].vlines(f_L,0,1,'g',lw=1.5,label=f'L-band ({len(f_L)}ch)')
axes[1][0].set_xlabel('Frequency (GHz)'); axes[1][0].set_ylabel('Channel')
axes[1][0].set_title('§3.1 ITU C+L WDM grid (100GHz)')
axes[1][0].legend(fontsize=8); axes[1][0].grid(True,alpha=0.3)

# GPU cluster scaling
axes[1][1].loglog(n_gpus, total_bw_needed/1e12,'bo-',ms=8,lw=2,label='BW needed (Tb/s)')
axes[1][1].loglog(n_gpus, n_fibers_needed,'rs--',ms=8,lw=2,label='Fibers needed')
axes[1][1].set_xlabel('GPU count'); axes[1][1].set_ylabel('Value')
axes[1][1].set_title('§3 AI cluster optical interconnect scaling')
axes[1][1].legend(fontsize=8); axes[1][1].grid(True,alpha=0.3)
for i,(ng,bw) in enumerate(zip(n_gpus, total_bw_needed)):
    axes[1][1].annotate(f'{ng}G',xy=(ng,bw/1e12),fontsize=7,ha='left')

# Modulation capacity bar
mod_names = list(mod_formats.keys())
caps_per_ch = [p['bits_per_sym']*p['baud']/1e9 for p in mod_formats.values()]
caps_total  = [c*(n_C+n_L)/1e3 for c in caps_per_ch]
x_m = np.arange(len(mod_names))
ax2b= axes[1][2].twinx()
axes[1][2].bar(x_m-0.2, caps_per_ch, width=0.35, color='blue', alpha=0.7, label='Gb/s/ch')
ax2b.bar(x_m+0.2,        caps_total,  width=0.35, color='green',alpha=0.7, label='Tb/s total')
axes[1][2].set_xticks(x_m); axes[1][2].set_xticklabels(mod_names,rotation=30,ha='right',fontsize=8)
axes[1][2].set_ylabel('Gb/s per channel',color='blue')
ax2b.set_ylabel('Tb/s total (C+L)',color='green')
axes[1][2].set_title('§3.1 Modulation format capacity')
axes[1][2].grid(True,alpha=0.3,axis='y')
plt.suptitle('🔬 §3: Photonic AI Datacenter', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §4 ⚙️ Vector Core / GEMM — Systolic Array · Roofline · Tiling

**§4.1 Roofline model:**
$\text{Perf} = \min(P_{peak},\ I \times BW_{mem})$, $I$ = arithmetic intensity (FLOP/byte)

**§4.2 Systolic array GEMM:** $C = A \times B$, $M\times N\times K$ — data flows like a wave

**§4.3 Cache tiling:** compute $C$ in tiles of size $T_m \times T_n$,
reuse $A$ row $T_k$ times → reduces DRAM traffic by $T_k$

**§4.4 GPU occupancy:** $\text{occ} = \text{active warps}/\text{max warps}$

In [ ]:
# §4 — Vector Core / GEMM

# §4.1 Roofline model for GPU / photonic ASIC
platforms = {
    'A100 SXM':      {'FLOP_peak':312e12,  'BW_peak':2e12,  'color':'green'},
    'H100 SXM':      {'FLOP_peak':1979e12, 'BW_peak':3.35e12,'color':'blue'},
    'MI300X':        {'FLOP_peak':1307e12, 'BW_peak':5.3e12, 'color':'red'},
    'Pi-core Photonic':{'FLOP_peak':100e12,'BW_peak':10e12,  'color':'purple'},
    'TPU v4':        {'FLOP_peak':275e12,  'BW_peak':1.2e12, 'color':'orange'},
}
I_arr = np.logspace(0, 4, 400)  # FLOP/byte

# §4.2 Tiled GEMM in numpy (blocked matrix multiply)
def gemm_tiled(A, B, T=64):
    M,K = A.shape; K2,N = B.shape
    assert K==K2
    C = np.zeros((M,N), dtype=A.dtype)
    for i in range(0,M,T):              # loop: tile M
        for j in range(0,N,T):          # loop: tile N
            for k in range(0,K,T):      # loop: tile K
                C[i:i+T, j:j+T] += A[i:i+T, k:k+T] @ B[k:k+T, j:j+T]
    return C

M,N,K = 512,512,512
A_rand = np.random.randn(M,K).astype(np.float32)
B_rand = np.random.randn(K,N).astype(np.float32)
t0 = time.perf_counter()
C_tiled = gemm_tiled(A_rand, B_rand, T=64)
t_tiled = time.perf_counter()-t0
C_ref   = A_rand @ B_rand
err_gemm = np.max(np.abs(C_tiled-C_ref))
FLOP_gemm= 2*M*N*K
TFLOPS_tiled = FLOP_gemm/t_tiled/1e12
print(f'§4.2 Tiled GEMM {M}x{K}x{N}: err={err_gemm:.2e}, {TFLOPS_tiled:.4f} TFLOPS')

# Tile size vs efficiency
tile_sizes = [16,32,64,128,256]
perf_tiles = []
for T in tile_sizes:                 # loop: tile size benchmark
    t0 = time.perf_counter()
    for _ in range(3): gemm_tiled(A_rand, B_rand, T=min(T,M))
    dt = (time.perf_counter()-t0)/3
    perf_tiles.append(FLOP_gemm/dt/1e9)  # GFLOPS

# §4.3 Arithmetic intensity for common ops
ops = {
    'Vector add':       {'I': 0.25,  'desc':'y=x+z'},
    'DAXPY':            {'I': 0.33,  'desc':'y=ax+y'},
    'Dot product':      {'I': 0.5,   'desc':'s=xᵀy'},
    'GEMV (M=N=4096)':  {'I': 1.0,   'desc':'y=Ax'},
    'GEMM (M=N=K=4096)':{'I': 2048,  'desc':'C=AB'},
    'Conv2d 3x3':       {'I': 4.5,   'desc':'3x3 conv'},
    'Attention (seq=2k)':{'I': 1000, 'desc':'QKᵀV'},
    'FFT (N=4096)':     {'I': 3.0,   'desc':'DFT'},
}

# §4.4 Warp occupancy model (NVIDIA Ampere)
max_warps    = 48   # per SM
max_regs_sm  = 65536
max_shmem_sm = 102400  # bytes
regs_per_warp_arr = np.arange(8,128,2)  # regs/thread, 32 threads/warp
threads_per_block = 256   # 8 warps/block
blocks_per_sm = np.floor(max_regs_sm / (regs_per_warp_arr*32*threads_per_block/32))
active_warps  = np.minimum(blocks_per_sm*8, max_warps)
occupancy     = active_warps/max_warps

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# Roofline
for pname,pp in platforms.items():    # loop: roofline curves
    I_ridge = pp['FLOP_peak']/pp['BW_peak']
    perf_rf = np.minimum(pp['FLOP_peak'], I_arr*pp['BW_peak'])/1e12
    axes[0][0].loglog(I_arr, perf_rf,'--',color=pp['color'],lw=2,label=pname)
    axes[0][0].axvline(I_ridge,color=pp['color'],lw=0.5,alpha=0.5)

# Plot ops as scatter
for op_name,op in ops.items():
    axes[0][0].axvline(op['I'],color='gray',lw=0.5,alpha=0.3)
    axes[0][0].text(op['I']*1.05,1,op_name[:8],fontsize=6,rotation=90,va='bottom',color='gray')
axes[0][0].set_xlabel('Arithmetic Intensity (FLOP/byte)')
axes[0][0].set_ylabel('Perf (TFLOPS)')
axes[0][0].set_title('§4.1 Roofline model (FP16/BF16)')
axes[0][0].legend(fontsize=6); axes[0][0].grid(True,alpha=0.3)

# Tile size vs performance
axes[0][1].semilogx(tile_sizes, perf_tiles,'bo-',ms=8,lw=2)
axes[0][1].set_xlabel('Tile size T'); axes[0][1].set_ylabel('GFLOPS')
axes[0][1].set_title(f'§4.3 GEMM tiling: {M}×{K}×{N} (float32, CPU)')
axes[0][1].grid(True,alpha=0.3)
for t,p in zip(tile_sizes,perf_tiles):
    axes[0][1].annotate(f'{p:.1f}',xy=(t,p),fontsize=7,ha='left',va='bottom')

# Systolic array visualization (8×8)
SZ = 8
ax_sys = axes[0][2]
ax_sys.set_xlim(-0.5,SZ-0.5); ax_sys.set_ylim(-0.5,SZ-0.5)
for i in range(SZ):              # loop: systolic PEs
    for j in range(SZ):
        col = plt.cm.Blues((i+j)/(2*SZ-2)*0.7+0.3)
        ax_sys.add_patch(Rectangle((j-0.45,i-0.45),0.9,0.9,
                                    facecolor=col,edgecolor='k',lw=0.5))
        ax_sys.text(j,i,'PE',ha='center',va='center',fontsize=6,color='white')
for i in range(SZ):
    ax_sys.annotate('',xy=(SZ-0.5,i),xytext=(SZ-1.5,i),
                    arrowprops=dict(arrowstyle='->',color='red',lw=0.8))
    ax_sys.annotate('',xy=(i,SZ-0.5),xytext=(i,SZ-1.5),
                    arrowprops=dict(arrowstyle='->',color='blue',lw=0.8))
ax_sys.set_title(f'§4.2 Systolic array ({SZ}×{SZ} PEs)\nRed=B data, Blue=A data')
ax_sys.set_xlabel('Column'); ax_sys.set_ylabel('Row')

# Occupancy vs registers
axes[1][0].plot(regs_per_warp_arr, occupancy*100,'r-',lw=2)
axes[1][0].axhline(80,ls='--',color='gray',lw=1.5,label='80% target')
axes[1][0].set_xlabel('Registers per thread')
axes[1][0].set_ylabel('Occupancy (%)')
axes[1][0].set_title('§4.4 GPU SM occupancy vs register use\n(Ampere, 256 threads/block)')
axes[1][0].legend(fontsize=8); axes[1][0].grid(True,alpha=0.3)

# Arithmetic intensity bar chart
op_names_I = list(ops.keys()); I_vals = [op['I'] for op in ops.values()]
colors_op  = plt.cm.RdYlGn(np.array(I_vals)/max(I_vals))
axes[1][1].barh(op_names_I, I_vals, color=colors_op, edgecolor='k', lw=0.5)
axes[1][1].axvline(100,ls='--',color='blue',lw=1.5,label='A100 ridge~100')
axes[1][1].set_xlabel('Arithmetic Intensity (FLOP/byte)')
axes[1][1].set_title('§4.3 Compute intensity: bound analysis')
axes[1][1].set_xscale('log'); axes[1][1].legend(fontsize=8)
axes[1][1].grid(True,alpha=0.3,axis='x')

# GEMM accuracy vs tile size
errs = []
for T in tile_sizes:               # loop: verify correctness per tile
    C_t = gemm_tiled(A_rand, B_rand, T=min(T,M))
    errs.append(np.max(np.abs(C_t-C_ref)))
axes[1][2].semilogy(tile_sizes, errs,'go-',ms=8,lw=2)
axes[1][2].set_xlabel('Tile size'); axes[1][2].set_ylabel('Max abs error')
axes[1][2].set_title(f'§4.2 GEMM correctness vs tile size\n(float32 rounding)')
axes[1][2].grid(True,alpha=0.3)
plt.suptitle('⚙️ §4: Vector Core / GEMM / Roofline', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §5 🌀 300k–500k Trajectory Integrals — C Structs → Python Dataclass → NumPy

**§5.1 Particle trajectory:** $\ddot{\mathbf{x}} = \mathbf{F}(\mathbf{x},\dot{\mathbf{x}},t)/m$

**§5.2 Vectorized RK4** (batch over particles, single loop over time steps):
$(k_1,k_2,k_3,k_4)$ computed on full $N\times 3$ arrays simultaneously

**§5.3 Action integral:**
$S = \int_0^T [\frac{1}{2}m|\dot{\mathbf{x}}|^2 - V(\mathbf{x})]\,dt$ — Hamilton's principle

**§5.4 C struct equivalent:**
```c
typedef struct { float x,y,z, vx,vy,vz; } Particle;
Particle particles[500000];
```

In [ ]:
# §5 — 300k-500k trajectory integrals

# §5.4 Python dataclass mirroring C struct
@dataclass
class Particle:
    x:  np.ndarray   # positions (N,3)
    v:  np.ndarray   # velocities (N,3)
    m:  float = 1.0
    q:  float = 0.0  # charge

@dataclass
class TrajectoryResult:
    positions:  np.ndarray  # (n_steps, N, 3)
    velocities: np.ndarray  # (n_steps, N, 3)
    times:      np.ndarray  # (n_steps,)
    actions:    np.ndarray  # (N,) — action integral per particle

# §5.2 Vectorized RK4 integrator
def rk4_batch(particles: Particle, F_func: Callable,
               dt: float, n_steps: int,
               record_every: int = 10) -> TrajectoryResult:
    x, v = particles.x.copy(), particles.v.copy()
    m = particles.m
    n_rec = n_steps // record_every
    pos_hist = np.empty((n_rec, *x.shape), dtype=np.float32)
    vel_hist = np.empty((n_rec, *x.shape), dtype=np.float32)
    t_hist   = np.empty(n_rec, dtype=np.float32)
    # Action accumulator: S_i += (0.5*m*|v|^2 - V(x)) * dt
    S_acc = np.zeros(x.shape[0], dtype=np.float64)
    t_cur = 0.0; rec_idx = 0
    for step in range(n_steps):           # loop: RK4 steps
        # k1
        a1 = F_func(x, v, t_cur) / m
        # k2
        x2 = x + 0.5*dt*v;  v2 = v + 0.5*dt*a1
        a2 = F_func(x2, v2, t_cur+0.5*dt) / m
        # k3
        x3 = x + 0.5*dt*v2; v3 = v + 0.5*dt*a2
        a3 = F_func(x3, v3, t_cur+0.5*dt) / m
        # k4
        x4 = x + dt*v3;     v4 = v + dt*a3
        a4 = F_func(x4, v4, t_cur+dt) / m
        x  = x + dt/6*(v + 2*v2 + 2*v3 + v4)
        v  = v + dt/6*(a1 + 2*a2 + 2*a3 + a4)
        t_cur += dt
        # Accumulate action (trapezoidal)
        KE = 0.5*m*np.sum(v**2, axis=1)
        V_pot = 0.5*m*np.sum(x**2, axis=1)   # harmonic V=0.5*k*r^2
        S_acc += (KE - V_pot)*dt
        if step % record_every == 0 and rec_idx < n_rec:
            pos_hist[rec_idx] = x.astype(np.float32)
            vel_hist[rec_idx] = v.astype(np.float32)
            t_hist[rec_idx]   = t_cur
            rec_idx += 1
    return TrajectoryResult(positions=pos_hist[:rec_idx],
                            velocities=vel_hist[:rec_idx],
                            times=t_hist[:rec_idx],
                            actions=S_acc)

# Forces: Lorentz-like (B field in z direction) + harmonic restoring
omega_c = 2.5  # cyclotron frequency
omega_0 = 1.0  # harmonic frequency
def F_Lorentz(x, v, t):
    # F = -k*x + q*(v × B),  B = B0*z_hat
    F_harm = -omega_0**2 * x
    # v × z_hat = (vy, -vx, 0)
    F_mag  = omega_c * np.stack([v[:,1], -v[:,0], np.zeros(v.shape[0])], axis=1)
    return F_harm + F_mag

# Batch of N particles
N_particles = 5000   # use 5k here; scales to 500k with same code
x0 = np.random.randn(N_particles,3).astype(np.float64)*0.5
v0 = np.random.randn(N_particles,3).astype(np.float64)*0.3
part = Particle(x=x0, v=v0, m=1.0, q=1.0)

t0_run = time.perf_counter()
result = rk4_batch(part, F_Lorentz, dt=0.01, n_steps=500, record_every=5)
t_run  = time.perf_counter()-t0_run

print(f'§5 Trajectory batch: N={N_particles}, steps=500, recorded={result.positions.shape[0]}')
print(f'   Runtime: {t_run:.3f}s  ({N_particles*500/t_run:.0f} particle-steps/s)')
print(f'   Action S: mean={result.actions.mean():.4f}, std={result.actions.std():.4f}')

# Extrapolate to 300k-500k particles
rate = N_particles*500/t_run
for Ntarget in [300_000, 500_000]:
    t_est = Ntarget*500/rate
    print(f'   Estimated {Ntarget:,} particles, 500 steps: {t_est:.1f}s ({t_est/60:.1f}min)')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# 3D trajectory of 5 particles
ax3d = fig.add_subplot(2,3,1,projection='3d')
colors3d = plt.cm.tab10(np.linspace(0,1,5))
for i in range(5):                   # loop: 5 example trajectories
    ax3d.plot(result.positions[:,i,0],
              result.positions[:,i,1],
              result.positions[:,i,2],
              color=colors3d[i],lw=1,alpha=0.8)
    ax3d.scatter(*result.positions[0,i],color=colors3d[i],s=30,marker='o')
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z')
ax3d.set_title(f'§5 Lorentz+harmonic\ntraj (5 of {N_particles})')

# Energy conservation
KE_hist = 0.5*(result.velocities**2).sum(axis=2)   # (n_rec, N)
PE_hist = 0.5*(result.positions **2).sum(axis=2)
E_hist  = KE_hist + PE_hist  # (n_rec, N)
E0      = E_hist[0]
dE_frac = (E_hist - E0[None,:]) / (E0[None,:] + 1e-10)
axes[0][1].plot(result.times, np.abs(dE_frac).mean(axis=1)*100,'r-',lw=2)
axes[0][1].set_xlabel('t (s)'); axes[0][1].set_ylabel('|ΔE/E₀| %')
axes[0][1].set_title('§5 RK4 energy conservation error\n(mean over particles)')
axes[0][1].grid(True,alpha=0.3)

# Action distribution
axes[0][2].hist(result.actions, bins=60, color='steelblue', edgecolor='k', lw=0.3)
axes[0][2].axvline(result.actions.mean(),color='r',ls='--',lw=2,
                    label=f'Mean={result.actions.mean():.2f}')
axes[0][2].set_xlabel('Action S'); axes[0][2].set_ylabel('Count')
axes[0][2].set_title('§5.3 Action integral S = ∫(KE-PE)dt')
axes[0][2].legend(fontsize=8); axes[0][2].grid(True,alpha=0.3)

# XY projection: density
all_x = result.positions[:,::10,0].ravel()
all_y = result.positions[:,::10,1].ravel()
h,xb,yb,im = axes[1][0].hist2d(all_x,all_y,bins=60,cmap='hot')
axes[1][0].set_xlabel('x'); axes[1][0].set_ylabel('y')
axes[1][0].set_title('§5 Particle density (x-y projection)')
plt.colorbar(im,ax=axes[1][0])

# Velocity distribution evolution
for i,t_idx in enumerate([0,10,50,99]):
    if t_idx < result.velocities.shape[0]:
        v_mag = np.linalg.norm(result.velocities[t_idx],axis=1)
        axes[1][1].hist(v_mag,bins=40,alpha=0.5,label=f't={result.times[min(t_idx,len(result.times)-1)]:.2f}')
axes[1][1].set_xlabel('|v| (m/s)'); axes[1][1].set_ylabel('Count')
axes[1][1].set_title('§5 Speed distribution evolution')
axes[1][1].legend(fontsize=7); axes[1][1].grid(True,alpha=0.3)

# Scaling projection
N_range = np.array([1e3,1e4,1e5,3e5,5e5,1e6])
t_proj  = N_range*500/rate
axes[1][2].loglog(N_range, t_proj,'b-o',ms=6,lw=2)
axes[1][2].axhline(60,ls='--',color='r',lw=1.5,label='1 min')
axes[1][2].axhline(600,ls='--',color='orange',lw=1.5,label='10 min')
axes[1][2].axvline(3e5,ls=':',color='gray',lw=1,label='300k target')
axes[1][2].set_xlabel('N particles'); axes[1][2].set_ylabel('Runtime (s, CPU)')
axes[1][2].set_title(f'§5 Scaling projection (CPU, {N_particles} measured)')
axes[1][2].legend(fontsize=7); axes[1][2].grid(True,alpha=0.3)
plt.suptitle('🌀 §5: 300k–500k Trajectory Integrals', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §6 💡 Free-Space EM Structs — Gaussian Beam · ABCD · Diffraction

**§6.1 ABCD matrix:** $\begin{pmatrix}q_2\\1\end{pmatrix} = M \begin{pmatrix}q_1\\1\end{pmatrix}$,
$q = z + iz_R$ (complex beam parameter)

**§6.2 Fraunhofer diffraction (aperture $a$):**
$U(\theta) = U_0 a\,\text{sinc}(a\sin\theta/\lambda)$

**§6.3 Huygens-Fresnel propagation (numerical):**
$U_2(x_2,y_2) = \frac{e^{ikz}}{i\lambda z}\iint U_1(x_1,y_1)e^{ik[(x_2-x_1)^2+(y_2-y_1)^2]/(2z)}\,dx_1\,dy_1$

In [ ]:
# §6 — Free-space EM + dataclass structs

# §6.1 ABCD matrix optics
@dataclass
class GaussianBeam:
    w0: float     # 1/e² beam waist (m)
    lam: float    # wavelength (m)
    z:  float = 0.0   # current position

    @property
    def zR(self):  return np.pi*self.w0**2/self.lam
    @property
    def q(self):   return self.z + 1j*self.zR
    def w(self):   return self.w0*np.sqrt(1+(self.z/self.zR)**2)
    def R(self):   return self.z*(1+(self.zR/self.z)**2) if self.z!=0 else np.inf
    def Gouy(self):return np.arctan(self.z/self.zR)

@dataclass
class OpticalElement:
    name: str
    M: np.ndarray  # 2x2 ABCD matrix

def propagate_q(q, M):
    A,B,C,D = M[0,0],M[0,1],M[1,0],M[1,1]
    return (A*q + B) / (C*q + D)

# ABCD matrices
def M_free(d):           return np.array([[1,d],[0,1]])
def M_thin_lens(f):      return np.array([[1,0],[-1/f,1]])
def M_curved_mirror(R):  return np.array([[1,0],[-2/R,1]])
def M_slab(n,d):         return np.array([[1,d/n],[0,1]])

# Optical setup: collimator → 5cm free → f=100mm lens → 10cm free
beam = GaussianBeam(w0=50e-6, lam=1064e-9, z=-0.05)
elements = [
    OpticalElement('Free 5cm',     M_free(0.05)),
    OpticalElement('Lens f=100mm', M_thin_lens(0.1)),
    OpticalElement('Free 10cm',    M_free(0.1)),
    OpticalElement('Lens f=50mm',  M_thin_lens(0.05)),
    OpticalElement('Free 5cm',     M_free(0.05)),
]
q_current = beam.q
positions  = [0.0]; widths = [beam.w0*1e6]; q_trace = [q_current]
z_current  = 0.0
for el in elements:                # loop: propagate through elements
    q_current = propagate_q(q_current, el.M)
    if 'Free' in el.name:
        d = el.M[0,1]
        z_current += d
    widths.append(np.sqrt(-beam.lam/(np.pi*np.imag(1/q_current)))*1e6)
    positions.append(z_current)
    q_trace.append(q_current)
    print(f'  After {el.name}: w = {widths[-1]:.1f} μm')

# §6.2 Fraunhofer diffraction: rectangular aperture
a_ap = 1e-3; b_ap = 0.5e-3; lam_diff = 632.8e-9; L_diff = 1.0  # m
theta_x = np.linspace(-0.01, 0.01, 400)
theta_y = np.linspace(-0.01, 0.01, 400)
TX,TY   = np.meshgrid(theta_x, theta_y)
sinc_x  = np.sinc(a_ap*TX/lam_diff)
sinc_y  = np.sinc(b_ap*TY/lam_diff)
I_diff  = (sinc_x*sinc_y)**2

# §6.3 Numerical Fresnel propagation (1D for speed)
N_f = 512; dx = 10e-6; z_prop = 0.05  # 5 cm
x1  = (np.arange(N_f)-N_f//2)*dx
k_f = 2*np.pi/lam_diff
# Input: slit
w_slit = 0.5e-3
U1     = np.where(np.abs(x1)<w_slit/2, 1.0+0j, 0.0)
# Angular spectrum propagation
fx = np.fft.fftfreq(N_f, dx)
H  = np.exp(1j*k_f*z_prop)*np.exp(-1j*np.pi*lam_diff*z_prop*(fx**2))
U2 = np.fft.ifft(np.fft.fft(U1)*H)
x2 = x1  # same grid (paraxial)

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# Beam width along optical path
axes[0][0].plot(positions, widths,'b-o',ms=8,lw=2)
for i,el in enumerate(elements):   # loop: mark elements
    if 'Lens' in el.name:
        axes[0][0].axvline(positions[i+1],color='r',ls='--',lw=1.5)
        axes[0][0].text(positions[i+1]+0.002, max(widths)*0.9, el.name,
                        fontsize=7,color='red',rotation=90,va='top')
axes[0][0].set_xlabel('z position (m)'); axes[0][0].set_ylabel('Beam radius (μm)')
axes[0][0].set_title('§6.1 Gaussian beam ABCD propagation')
axes[0][0].grid(True,alpha=0.3)

# Complex beam parameter trace (Argand)
q_re = [q.real for q in q_trace]; q_im = [q.imag for q in q_trace]
axes[0][1].plot(q_re, q_im,'go-',ms=8,lw=2)
for i,(qr,qi) in enumerate(zip(q_re,q_im)):
    label = ['Input']+[el.name for el in elements]
    axes[0][1].annotate(label[i][:8],(qr,qi),fontsize=6,
                         textcoords='offset points',xytext=(4,4))
axes[0][1].set_xlabel('Re[q] (m)'); axes[0][1].set_ylabel('Im[q] = z_R (m)')
axes[0][1].set_title('§6.1 Complex beam param q trace')
axes[0][1].grid(True,alpha=0.3); axes[0][1].axhline(0,color='k',lw=0.5)

# Fraunhofer 2D
im_f2d = axes[0][2].imshow(I_diff, extent=[theta_x[0]*1e3,theta_x[-1]*1e3,
                                             theta_y[0]*1e3,theta_y[-1]*1e3],
                             cmap='hot', origin='lower', aspect='auto')
axes[0][2].set_xlabel('θx (mrad)'); axes[0][2].set_ylabel('θy (mrad)')
axes[0][2].set_title(f'§6.2 Fraunhofer: {a_ap*1e3:.1f}×{b_ap*1e3:.1f}mm aperture')
plt.colorbar(im_f2d,ax=axes[0][2])

# Fresnel propagation 1D
axes[1][0].plot(x1*1e3, np.abs(U1),'b-',lw=2,label='Input (z=0)',alpha=0.7)
axes[1][0].plot(x2*1e3, np.abs(U2)/np.abs(U2).max(),'r-',lw=2,
                label=f'z={z_prop*100:.0f}cm (Fresnel)')
axes[1][0].set_xlabel('x (mm)'); axes[1][0].set_ylabel('|U| (normalized)')
axes[1][0].set_title(f'§6.3 Fresnel: slit {w_slit*1e3:.1f}mm, z={z_prop*100:.0f}cm')
axes[1][0].legend(fontsize=8); axes[1][0].grid(True,alpha=0.3)
axes[1][0].set_xlim(-3,3)

# Gouy phase
z_Gouy = np.linspace(-5*beam.zR, 5*beam.zR, 300)
phi_Gouy = np.arctan(z_Gouy/beam.zR)
axes[1][1].plot(z_Gouy*1e3, np.degrees(phi_Gouy),'m-',lw=2)
axes[1][1].axhline(-90,ls='--',color='k',lw=1); axes[1][1].axhline(90,ls='--',color='k',lw=1)
axes[1][1].set_xlabel('z (mm)'); axes[1][1].set_ylabel('Gouy phase (deg)')
axes[1][1].set_title(f'§6 Gouy phase shift (zR={beam.zR*1e3:.1f}mm)')
axes[1][1].grid(True,alpha=0.3)

# Intensity on 1D slit (log scale)
I_slit = np.abs(U2)**2
axes[1][2].semilogy(x2*1e3, I_slit/I_slit.max(),'r-',lw=2)
axes[1][2].set_xlabel('x (mm)'); axes[1][2].set_ylabel('|U|² (norm, log)')
axes[1][2].set_title('§6.3 Fresnel intensity (log scale)')
axes[1][2].set_xlim(-5,5); axes[1][2].grid(True,alpha=0.3)
plt.suptitle('💡 §6: Free-Space EM — Gaussian Beam + Diffraction', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §7 🔴 Lasers + Spectroscopy — Voigt · Beer-Lambert · HPC TOP500

**§7.1 Voigt lineshape:**
$V(\nu,\nu_0) = \int G(\nu',\Delta\nu_G) \cdot L(\nu-\nu',\Delta\nu_L)\,d\nu'$
(convolution of Gaussian Doppler broadening + Lorentzian pressure broadening)

**§7.2 HITRAN-style absorption:** $\sigma(\nu) = S \cdot V(\nu,\nu_0)$, $I = I_0 e^{-\sigma n L}$

**§7.3 TOP500 HPC + photonic interconnect need:**
Total switch bandwidth ∝ N² grows faster than electrical can deliver → photonics required

In [ ]:
# §7 — Lasers + spectroscopy + TOP500

# §7.1 Voigt lineshape (wofz implementation = exact complex error function)
def voigt(nu, nu0, fwhm_G, fwhm_L):
    sigma_G = fwhm_G/(2*np.sqrt(2*np.log(2)))  # Gaussian sigma from FWHM
    gamma_L = fwhm_L/2                           # Lorentzian half-width
    z       = ((nu - nu0) + 1j*gamma_L) / (sigma_G*np.sqrt(2))
    return np.real(wofz(z)) / (sigma_G*np.sqrt(2*np.pi))

# CO2 absorption at 4.26 μm (ν3 band fundamental)
nu0_CO2  = 2349.0   # cm⁻¹ (HITRAN)
S_CO2    = 2.4e-17  # cm/molecule line strength
T_gas    = 296.0    # K
P_gas    = 101325.0 # Pa
c_mol    = 2.998e10 # cm/s
h_pl     = 6.626e-34; k_B=1.381e-23; N_A=6.022e23
M_CO2    = 44e-3    # kg/mol
m_CO2    = M_CO2/N_A

fwhm_D   = nu0_CO2/c_mol * np.sqrt(8*np.log(2)*k_B*T_gas/m_CO2)  # Doppler cm⁻¹
P_air    = 101325.0; gamma_air = 0.064  # cm⁻¹/atm (CO2 air-broadening coeff)
fwhm_L_CO2 = 2*gamma_air*(P_gas/101325.0)  # Lorentzian FWHM
print(f'§7.1 CO2 @4.26μm: Doppler FWHM={fwhm_D*1e3:.3f}×10⁻³ cm⁻¹, Lorentzian={fwhm_L_CO2:.4f}cm⁻¹')

nu_arr  = np.linspace(nu0_CO2-0.3, nu0_CO2+0.3, 2000)
V_CO2   = voigt(nu_arr, nu0_CO2, fwhm_D, fwhm_L_CO2)
V_G     = (1/(fwhm_D/2.355*np.sqrt(2*np.pi))*np.exp(-0.5*((nu_arr-nu0_CO2)/(fwhm_D/2.355))**2))
V_L     = (fwhm_L_CO2/2)/(np.pi*((nu_arr-nu0_CO2)**2+(fwhm_L_CO2/2)**2))

# §7.2 Beer-Lambert: CO2 in atmosphere
n_CO2   = 420e-6 * N_A * P_gas / (8.314 * T_gas) / 1e6  # molecules/cm³ (420 ppm)
sigma_nu= S_CO2 * V_CO2   # cross-section cm²/molecule
L_path  = np.linspace(0, 100, 200)  # cm path length
# Transmission at line center vs path
T_center= np.exp(-sigma_nu[len(nu_arr)//2]*n_CO2*L_path)
# Transmission spectrum at L=10cm
T_spec  = np.exp(-sigma_nu*n_CO2*10.0)
print(f'§7.2 CO2 at 420ppm: σ_peak={sigma_nu.max():.2e}cm², T(L=10cm)={T_spec.min():.4f}')

# Laser linewidth comparison
lasers = {
    'Free-running diode': {'FWHM_Hz': 10e6,    'color':'red'},
    'ECDL (grating)':     {'FWHM_Hz': 100e3,   'color':'blue'},
    'Nd:YAG (CW)':        {'FWHM_Hz': 10e3,    'color':'green'},
    'OFC tooth':          {'FWHM_Hz': 1e0,     'color':'purple'},
    'H-maser ref':        {'FWHM_Hz': 0.001,   'color':'orange'},
}

# §7.3 TOP500 HPC data (historical trend + photonic need)
top500_years = np.array([1993,1996,1999,2002,2005,2008,2011,2014,2017,2020,2023,2025])
top500_Gflops= np.array([59.7, 368, 2379, 35860, 280600, 1026000, 10510000,
                          33863000, 93014700, 415530000, 1194000000, 2000000000])  # GFlops

# Bisection bandwidth needed scales as N_nodes^(7/4) (all-to-all approx)
# For top system: ~50k nodes × 800 Gb/s NVLink = total BW
n_nodes = np.array([140, 336, 9632, 9632*5, 65536, 131072, 705024,
                    4608, 27360, 4608, 11088, 40000])
bw_needed_Pbps = n_nodes**1.5 * 400e9 / 1e15  # rough scaling

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# Lineshape comparison
axes[0][0].plot(nu_arr-nu0_CO2, V_CO2,'k-',lw=2,label='Voigt')
axes[0][0].plot(nu_arr-nu0_CO2, V_G,'b--',lw=1.5,label='Gaussian (Doppler)')
axes[0][0].plot(nu_arr-nu0_CO2, V_L,'r:',lw=1.5,label='Lorentzian (pressure)')
axes[0][0].set_xlabel('Δν (cm⁻¹)'); axes[0][0].set_ylabel('Lineshape (cm)')
axes[0][0].set_title(f'§7.1 CO2 4.26μm Voigt lineshape\nT={T_gas}K, P=1atm')
axes[0][0].legend(fontsize=8); axes[0][0].grid(True,alpha=0.3)

# Absorption spectrum
axes[0][1].plot(nu_arr, T_spec,'g-',lw=2)
axes[0][1].fill_between(nu_arr, T_spec, alpha=0.2, color='green')
axes[0][1].set_xlabel('ν (cm⁻¹)'); axes[0][1].set_ylabel('Transmittance')
axes[0][1].set_title('§7.2 CO2 absorption spectrum (L=10cm, 420ppm)')
axes[0][1].grid(True,alpha=0.3)

# Beer-Lambert: T vs path
axes[0][2].plot(L_path, T_center,'r-',lw=2)
axes[0][2].axhline(np.exp(-1),ls='--',color='k',lw=1,label='1/e')
axes[0][2].set_xlabel('Path length L (cm)'); axes[0][2].set_ylabel('Transmittance (line center)')
axes[0][2].set_title('§7.2 Beer-Lambert at line center\n(CO2 420ppm)')
axes[0][2].legend(fontsize=8); axes[0][2].grid(True,alpha=0.3)

# Laser linewidth
names_lw = list(lasers.keys()); fwhms = [v['FWHM_Hz'] for v in lasers.values()]
colors_lw= [v['color'] for v in lasers.values()]
axes[1][0].barh(names_lw, fwhms, color=colors_lw, alpha=0.8, edgecolor='k')
axes[1][0].set_xscale('log'); axes[1][0].set_xlabel('Linewidth FWHM (Hz)')
axes[1][0].set_title('§7 Laser linewidth comparison'); axes[1][0].grid(True,alpha=0.3,axis='x')

# TOP500 performance trend
axes[1][1].semilogy(top500_years, top500_Gflops/1e6,'b-o',ms=6,lw=2,label='#1 system (TFLOPS)')
axes[1][1].set_xlabel('Year'); axes[1][1].set_ylabel('Performance (TFLOPS)')
axes[1][1].set_title('§7.3 TOP500 #1 system (1993–2025)')
axes[1][1].legend(fontsize=8); axes[1][1].grid(True,alpha=0.3)
# Annotate doubling time
axes[1][1].text(2010, 1e6, 'Doubling\nevery ~14mo', fontsize=8, color='gray')

# Photonic interconnect need
axes[1][2].semilogy(top500_years, bw_needed_Pbps,'r-s',ms=6,lw=2,label='BW needed (Pb/s)')
axes[1][2].axhline(0.001,ls='--',color='gray',lw=1,label='1 Tb/s (electrical)')
axes[1][2].axhline(1,ls='--',color='green',lw=1,label='1 Pb/s (photonic)')
axes[1][2].set_xlabel('Year'); axes[1][2].set_ylabel('Bisection BW (Pb/s)')
axes[1][2].set_title('§7.3 HPC interconnect BW need\n→ photonics required')
axes[1][2].legend(fontsize=7); axes[1][2].grid(True,alpha=0.3)
plt.suptitle('🔴 §7: Lasers + Spectroscopy + TOP500 HPC', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §8 📐 Fixed Points + Tilt Dynamic Equilibrium

**§8.1 1D ODE fixed points:** $\dot x = f(x)$, fixed points at $f(x^*)=0$;
stable if $f'(x^*)<0$, unstable if $f'(x^*)>0$

**§8.2 2D phase portrait:** van der Pol oscillator
$\ddot x - \mu(1-x^2)\dot x + x = 0$

**§8.3 Tilt stability (inverted pendulum):**
$\ddot\theta = (g/L)\sin\theta - (b/mL^2)\dot\theta$; stable fixed point at $\theta=0$
with control $u = -k_p\theta - k_d\dot\theta$

**§8.4 Lyapunov function:** $V(x) > 0$, $\dot V < 0$ → globally stable

In [ ]:
# §8 — Fixed points + dynamic equilibrium

# §8.1 Fixed points: 1D pitchfork bifurcation
# dx/dt = r*x - x^3, fixed points at x*=0 and x*=±sqrt(r)
r_arr = np.linspace(-2, 2, 300)
x_sym = np.linspace(-2, 2, 400)
fig_bif, ax_bif = plt.subplots(1,1,figsize=(5,4))
for r in np.linspace(-2,2,30):  # loop: bifurcation diagram
    roots = [0.0]
    if r>0: roots += [np.sqrt(r), -np.sqrt(r)]
    for x_star in roots:
        fp_prime = r - 3*x_star**2
        col = 'blue' if fp_prime<0 else 'red'
        ax_bif.plot(r, x_star, 'o', color=col, ms=4, alpha=0.7)
ax_bif.axhline(0,color='k',lw=0.5); ax_bif.axvline(0,color='k',lw=0.5)
ax_bif.set_xlabel('r (bifurcation param)'); ax_bif.set_ylabel('x*')
ax_bif.set_title('§8.1 Pitchfork bifurcation: ẋ=rx-x³\nBlue=stable, Red=unstable')
ax_bif.grid(True,alpha=0.3)

# §8.2 Van der Pol: 2D phase portrait
mu_vdP = 1.5
def vdP(t, y):
    x,v = y
    return [v, mu_vdP*(1-x**2)*v - x]

# Limit cycle by long integration
sol_lc = solve_ivp(vdP,[0,50],[0.1,0.1],t_eval=np.linspace(30,50,2000),method='RK45')

# Grid of trajectories
x_grid = np.linspace(-4,4,20); v_grid = np.linspace(-6,6,20)
XX,VV  = np.meshgrid(x_grid,v_grid)
dX = VV; dV = mu_vdP*(1-XX**2)*VV - XX
speed = np.sqrt(dX**2+dV**2)+1e-10

# §8.3 Inverted pendulum with PD control
g_pen=9.81; L_pen=0.5; m_pen=0.1; b_pen=0.05
kp=20.0; kd=5.0   # PD gains

def pendulum_controlled(t,y):
    theta,omega = y
    u = -kp*theta - kd*omega   # PD control torque
    dthetadt = omega
    domegadt  = (g_pen/L_pen)*np.sin(theta) - (b_pen/(m_pen*L_pen**2))*omega + u/(m_pen*L_pen**2)
    return [dthetadt, domegadt]

def pendulum_free(t,y):
    theta,omega = y
    return [omega, (g_pen/L_pen)*np.sin(theta) - (b_pen/(m_pen*L_pen**2))*omega]

# Simulate from various initial conditions
theta0_arr = [0.3, 0.8, 1.2, -0.5, -1.0]
t_eval_pen = np.linspace(0,5,1000)
results_ctrl = []
results_free = []
for th0 in theta0_arr:    # loop: initial conditions
    sol_c = solve_ivp(pendulum_controlled,[0,5],[th0,0],t_eval=t_eval_pen)
    sol_f = solve_ivp(pendulum_free,     [0,5],[th0,0],t_eval=t_eval_pen)
    results_ctrl.append(sol_c)
    results_free.append(sol_f)

# §8.4 Lyapunov: V(x,v) = 0.5*x^2 + 0.5*v^2 for linear system ẋ=v, v̇=-x-v
x_ly = np.linspace(-3,3,100); v_ly=np.linspace(-3,3,100)
XL,VL = np.meshgrid(x_ly,v_ly)
V_lyap= 0.5*XL**2 + 0.5*VL**2
dVdt  = XL*VL + VL*(-XL-VL)  # dV/dt = x*ẋ + v*v̇

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,9))

# Bifurcation
# Already drawn in fig_bif — embed data
for r in np.linspace(-2,2,80):    # loop: redo for main fig
    roots = [0.0]
    if r>0: roots += [np.sqrt(r), -np.sqrt(r)]
    for x_star in roots:
        fp_prime = r - 3*x_star**2
        col = 'blue' if fp_prime<0 else 'red'
        axes[0][0].plot(r, x_star, 'o', color=col, ms=3, alpha=0.6)
axes[0][0].axhline(0,color='k',lw=0.5); axes[0][0].axvline(0,color='k',lw=0.5)
axes[0][0].set_xlabel('r'); axes[0][0].set_ylabel('x*')
axes[0][0].set_title('§8.1 Pitchfork bifurcation ẋ=rx-x³')
axes[0][0].grid(True,alpha=0.3)

# Van der Pol phase portrait
axes[0][1].streamplot(x_grid,v_grid,dX,dV,density=1.0,
                      color=np.log(speed),cmap='Blues',linewidth=0.8)
axes[0][1].plot(sol_lc.y[0], sol_lc.y[1],'r-',lw=2.5,label='Limit cycle')
axes[0][1].plot(0,0,'ko',ms=8,label='Unstable FP')
axes[0][1].set_xlabel('x'); axes[0][1].set_ylabel('ẋ')
axes[0][1].set_title(f'§8.2 Van der Pol μ={mu_vdP} phase portrait')
axes[0][1].legend(fontsize=8); axes[0][1].grid(True,alpha=0.3)
axes[0][1].set_xlim(-4,4); axes[0][1].set_ylim(-6,6)

# Inverted pendulum: controlled vs free
for i,(sc,sf) in enumerate(zip(results_ctrl,results_free)):  # loop: trajectories
    axes[0][2].plot(sc.t, np.degrees(sc.y[0]),'b-',alpha=0.7,lw=1.5)
    axes[0][2].plot(sf.t, np.degrees(sf.y[0]),'r--',alpha=0.5,lw=1)
axes[0][2].axhline(0,color='k',lw=1,ls=':')
axes[0][2].set_xlabel('t (s)'); axes[0][2].set_ylabel('θ (deg)')
axes[0][2].set_title(f'§8.3 Inverted pendulum: PD (blue) vs free (red)\nkp={kp}, kd={kd}')
axes[0][2].grid(True,alpha=0.3)

# Lyapunov V(x,v)
ax_ly = axes[1][0]
cs = ax_ly.contourf(XL,VL,V_lyap,20,cmap='YlOrRd')
plt.colorbar(cs,ax=ax_ly)
ax_ly.contour(XL,VL,dVdt,levels=[0],colors='k',linewidths=2)
# dV/dt negative everywhere → globally stable
ax_ly.set_xlabel('x'); ax_ly.set_ylabel('v')
ax_ly.set_title('§8.4 Lyapunov V=0.5(x²+v²)\nBlack=dV/dt=0 boundary')

# Phase portrait: controlled pendulum
for sc in results_ctrl:          # loop: pendulum phase portrait
    axes[1][1].plot(np.degrees(sc.y[0]), np.degrees(sc.y[1]),'b-',alpha=0.7,lw=1.5)
axes[1][1].plot(0,0,'go',ms=12,label='Stable FP (θ=0,ω=0)')
axes[1][1].set_xlabel('θ (deg)'); axes[1][1].set_ylabel('ω (deg/s)')
axes[1][1].set_title('§8.3 Pendulum phase portrait (PD control)')
axes[1][1].legend(fontsize=8); axes[1][1].grid(True,alpha=0.3)

# Tilt response vs gain (kp)
kp_arr = np.linspace(0,50,20)
max_overshoot = []
for kp_i in kp_arr:             # loop: gain sweep
    def pen_kp(t,y): return [y[1],(g_pen/L_pen)*np.sin(y[0])-(b_pen/(m_pen*L_pen**2))*y[1]+
                              (-kp_i*y[0]-kd*y[1])/(m_pen*L_pen**2)]
    try:
        sol_i = solve_ivp(pen_kp,[0,5],[0.5,0],t_eval=np.linspace(0,5,500),method='RK45')
        if sol_i.success: max_overshoot.append(np.max(np.abs(np.degrees(sol_i.y[0]))))
        else: max_overshoot.append(np.nan)
    except: max_overshoot.append(np.nan)
axes[1][2].plot(kp_arr, max_overshoot,'m-o',ms=5,lw=2)
axes[1][2].axhline(5,ls='--',color='r',lw=1.5,label='5° overshoot limit')
axes[1][2].set_xlabel('kp (proportional gain)'); axes[1][2].set_ylabel('Max |θ| (deg)')
axes[1][2].set_title(f'§8.3 Pendulum: overshoot vs kp (kd={kd})')
axes[1][2].legend(fontsize=8); axes[1][2].grid(True,alpha=0.3)
plt.suptitle('📐 §8: Fixed Points + Tilt Dynamic Equilibrium', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
plt.close(fig_bif)

---
## §9 🌐 IP/DNS/DOM Agents — Network Topology · Socket DNS · HTML Parse

**§9.1 IP address structure:**
IPv4: 32-bit; IPv6: 128-bit; CIDR: `192.168.1.0/24` → 254 usable hosts

**§9.2 DNS resolution chain:**
stub resolver → recursive resolver → root NS → TLD NS → authoritative NS

**§9.3 DOM agent** (pure Python `html.parser`): parse HTML, extract structure,
count tags, find links — no external deps, runs offline.

**§9.4 Network agent pattern:** `run(query) → {dns_records, ip_info, dom_summary}`

In [ ]:
# §9 — IP/DNS/DOM Agents (stdlib-only, works offline)
import ipaddress, socket, html.parser, urllib.request, struct as st

# §9.1 IP address tools
def ip_info(addr_str):
    try:
        addr = ipaddress.ip_address(addr_str)
        return {'version':addr.version,'is_private':addr.is_private,
                'is_loopback':addr.is_loopback,'packed':addr.packed.hex()}
    except: return {'error': f'Invalid: {addr_str}'}

def cidr_info(cidr_str):
    net  = ipaddress.ip_network(cidr_str, strict=False)
    return {'network':str(net.network_address),'broadcast':str(net.broadcast_address),
            'num_hosts':net.num_addresses-2,'prefix':net.prefixlen,
            'first_host':str(list(net.hosts())[0]) if net.num_addresses>2 else 'N/A',
            'last_host': str(list(net.hosts())[-1])if net.num_addresses>2 else 'N/A'}

test_ips   = ['192.168.1.10','10.0.0.1','8.8.8.8','::1','2001:db8::1','240.0.0.1']
test_cidrs = ['192.168.1.0/24','10.0.0.0/8','172.16.0.0/12','192.168.100.0/27']

print('§9.1 IP address analysis:')
for ip in test_ips:        # loop: IP info
    info = ip_info(ip)
    print(f'  {ip:20s}: v{info.get("version","?")} private={info.get("is_private","?")}')

print('\n§9.1 CIDR network info:')
for cidr in test_cidrs:    # loop: CIDR
    c = cidr_info(cidr)
    print(f'  {cidr:22s}: {c["num_hosts"]} hosts, {c["first_host"]}–{c["last_host"]}')

# §9.2 DNS resolution (stdlib — works if network available, graceful fallback)
def dns_lookup(hostname):
    try:
        results = socket.getaddrinfo(hostname, None)
        ips = list(set(r[4][0] for r in results))
        fqdn = socket.getfqdn(hostname)
        rev  = []
        for ip in ips[:2]:     # loop: reverse lookup
            try: rev.append(socket.gethostbyaddr(ip)[0])
            except: rev.append('N/A')
        return {'hostname':hostname,'ips':ips,'fqdn':fqdn,'reverse':rev}
    except socket.gaierror as e:
        return {'hostname':hostname,'error':str(e),'ips':[]}

# Try a few resolutions (may need network)
hosts_to_resolve = ['localhost','127.0.0.1']
print('\n§9.2 DNS resolution:')
for host in hosts_to_resolve:  # loop: DNS
    r = dns_lookup(host)
    if 'error' not in r:
        print(f'  {host:20s} → {r["ips"]}')
    else:
        print(f'  {host:20s} → (offline: {r["error"][:40]})')

# §9.3 DOM Agent (html.parser)
class DOMAgent(html.parser.HTMLParser):
    def __init__(self):
        super().__init__()
        self.tag_counts: Dict[str,int] = {}
        self.links: List[str] = []
        self.scripts: List[str] = []
        self.text_chunks: List[str] = []
        self.depth = 0; self.max_depth = 0

    def handle_starttag(self, tag, attrs):
        self.tag_counts[tag] = self.tag_counts.get(tag,0)+1
        self.depth += 1; self.max_depth = max(self.depth, self.max_depth)
        attrs_d = dict(attrs)
        if tag == 'a' and 'href' in attrs_d:
            self.links.append(attrs_d['href'])
        if tag == 'script' and 'src' in attrs_d:
            self.scripts.append(attrs_d['src'])

    def handle_endtag(self, tag): self.depth -= 1

    def handle_data(self, data):
        t = data.strip()
        if t: self.text_chunks.append(t[:80])

    def summary(self):
        return {'n_tags': sum(self.tag_counts.values()),
                'unique_tags': len(self.tag_counts),
                'max_depth': self.max_depth,
                'n_links': len(self.links),
                'n_scripts': len(self.scripts),
                'top_tags': sorted(self.tag_counts.items(),key=lambda x:-x[1])[:5],
                'sample_links': self.links[:5],
                'tag_counts': self.tag_counts}

# Test DOM agent on synthetic HTML
test_html = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Photonic Computing Platform</title>
  <script src="/static/main.js"></script>
  <script src="/static/torch.min.js"></script>
</head>
<body>
  <header><nav>
    <a href="/">Home</a> <a href="/modules">Modules</a>
    <a href="/docs">Docs</a> <a href="/api">API</a>
  </nav></header>
  <main>
    <h1>GS Phase Recovery</h1>
    <section id="pipeline">
      <h2>Dispersion Assisted Pipeline</h2>
      <ul>
        <li><a href="/modules/gs_core">gs_core.py</a></li>
        <li><a href="/modules/gs_fno">gs_fno.py</a></li>
        <li><a href="/modules/rogue_guard">RogueGuard</a></li>
      </ul>
      <p>TD-GS convergence with unit-amplitude constraint and |D|>=5000.</p>
    </section>
    <section id="photonics">
      <h2>Photonic Interconnect</h2>
      <div class="card"><p>WDM C+L band: 80 channels × 400 Gb/s = 32 Tb/s</p></div>
      <div class="card"><p>Energy: 0.3 pJ/bit target</p></div>
    </section>
  </main>
  <footer><p>© 2026 Dispersion-Assisted-GS</p></footer>
</body>
</html>'''

agent = DOMAgent()
agent.feed(test_html)
s = agent.summary()
print(f'\n§9.3 DOM Agent parse:')
print(f'  Tags: {s["n_tags"]} total, {s["unique_tags"]} unique, max depth={s["max_depth"]}')
print(f'  Links: {s["n_links"]}: {s["sample_links"]}')
print(f'  Scripts: {agent.scripts}')
print(f'  Top tags: {s["top_tags"]}')

# §9.4 Network agent: compose IP + DNS + DOM
@register('network_agent')
class NetworkAgent(Module):
    def run(self, inp):
        url     = inp.get('url', 'http://localhost')
        html_src= inp.get('html', '')
        hosts   = inp.get('hosts', ['localhost'])
        ip_results  = {ip: ip_info(ip) for ip in inp.get('ips',['127.0.0.1'])}
        dns_results = {h: dns_lookup(h) for h in hosts}
        dom_result  = {}
        if html_src:
            ag = DOMAgent(); ag.feed(html_src)
            dom_result = ag.summary()
        return {'ip_analysis': ip_results, 'dns': dns_results, 'dom': dom_result}
    def describe(self): return 'Network agent: IP + DNS + DOM parsing'

net_result = _REGISTRY['network_agent']().run({
    'ips':   ['192.168.1.1','10.0.0.1','8.8.8.8'],
    'hosts': ['localhost'],
    'html':  test_html
})
print(f'\n§9.4 Network agent: {len(net_result["ip_analysis"])} IPs, {len(net_result["dom"]["top_tags"])} tag types')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(15,4))

# CIDR subnet size visualization
cidr_list  = [f'10.0.0.0/{p}' for p in range(8,29)]
hosts_list = [ipaddress.ip_network(c,strict=False).num_addresses-2 for c in cidr_list]
prefix_list= list(range(8,29))
axes[0].semilogy(prefix_list, hosts_list,'b-o',ms=5,lw=2)
axes[0].set_xlabel('Prefix length /n'); axes[0].set_ylabel('Usable hosts')
axes[0].set_title('§9.1 CIDR usable hosts vs prefix\n(Class A 10.0.0.0/n)')
axes[0].grid(True,alpha=0.3)
for p,h in zip([8,16,24,27],[2**24-2,2**16-2,254,30]):
    axes[0].annotate(f'/{p}: {h}',xy=(p,h),fontsize=8,ha='left',
                      textcoords='offset points',xytext=(3,3))

# DNS resolution chain diagram
ax_dns = axes[1]; ax_dns.axis('off')
dns_chain = ['Browser\nstub resolver','OS\nrecursive','Root NS\n(13 servers)',
             'TLD NS\n(.com/.edu)','Auth NS\n(final answer)']
dns_cols  = ['#E3F2FD','#BBDEFB','#90CAF9','#64B5F6','#42A5F5']
for i,(label,col) in enumerate(zip(dns_chain,dns_cols)):
    y = 0.8 - i*0.18
    ax_dns.add_patch(plt.Rectangle((0.1,y-0.07),0.8,0.14,
                                    facecolor=col,edgecolor='k',lw=1.5))
    ax_dns.text(0.5,y,label,ha='center',va='center',fontsize=8,fontweight='bold')
    if i<len(dns_chain)-1:
        ax_dns.annotate('',xy=(0.5,y-0.08),xytext=(0.5,y-0.07),
                         arrowprops=dict(arrowstyle='<->',color='k',lw=1.5))
ax_dns.set_title('§9.2 DNS Resolution Chain')

# DOM tag distribution
tag_counts = s['tag_counts']
tag_names = list(tag_counts.keys()); tag_vals = list(tag_counts.values())
colors_dom = plt.cm.Set3(np.linspace(0,1,len(tag_names)))
axes[2].bar(tag_names, tag_vals, color=colors_dom, edgecolor='k', lw=0.5)
axes[2].set_ylabel('Count'); axes[2].set_title(f'§9.3 DOM tag distribution\n({s["n_tags"]} total)')
axes[2].tick_params(axis='x',rotation=45)
plt.suptitle('🌐 §9: IP/DNS/DOM Agents', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('\nAll 9 sections complete. Notebook ready.')